In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
from copy import deepcopy

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor

from sklearn.linear_model import BayesianRidge, Lasso
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

TRAIN_CSV = "train.csv"
TEST_CSV  = "test.csv"

B = 1000               
SEED_BASE = 42         
LOO = LeaveOneOut()

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def metrics_tuple(y_true, y_pred):
    return (r2_score(y_true, y_pred), rmse(y_true, y_pred), mean_absolute_error(y_true, y_pred))

def paired_bootstrap_ci(y_true, y_pred, B=B, seed=SEED_BASE):
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2s, rmses, maes = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_pred[idx]
        r2s.append(r2_score(yt, yp))
        rmses.append(mean_squared_error(yt, yp, squared=False))
        maes.append(mean_absolute_error(yt, yp))
    return {
        "r2_ci": np.percentile(r2s, [2.5, 97.5]),
        "rmse_ci": np.percentile(rmses, [2.5, 97.5]),
        "mae_ci": np.percentile(maes, [2.5, 97.5])
    }

def pretty_point(name, y_true, y_pred):
    r2, r_rmse, r_mae = metrics_tuple(y_true, y_pred)
    print(f"{name:<6}| R²={r2:.3f}  RMSE={r_rmse:.3f}  MAE={r_mae:.3f}")
    return (r2, r_rmse, r_mae)

def pretty_with_ci(name, pt_tuple, ci_dict):
    r2, r_rmse, r_mae = pt_tuple
    r2_l, r2_h = ci_dict["r2_ci"]
    rmse_l, rmse_h = ci_dict["rmse_ci"]
    mae_l, mae_h = ci_dict["mae_ci"]
    print(f"{name:<6}| R²={r2:.3f} (95% CI {r2_l:.3f}–{r2_h:.3f}) | "
          f"RMSE={r_rmse:.3f} (95% CI {rmse_l:.3f}–{rmse_h:.3f}) | "
          f"MAE={r_mae:.3f} (95% CI {mae_l:.3f}–{mae_h:.3f})")

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

y_train = train_df["Liver_regeneration_rate"].values
y_test  = test_df["Liver_regeneration_rate"].values

X_train_raw = train_df.drop(columns=["Liver_regeneration_rate"])
X_test_raw  = test_df.drop(columns=["Liver_regeneration_rate"])

# Identify columns
cat_cols = ["Time_points"]
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

def create_preprocess():
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    num_pipe = Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=False)),
        ("scale", StandardScaler())
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", ohe,      cat_cols)
    ])

MODEL_PARAMS = {
    "BR": {"alpha_1":1.0, "alpha_2":1e-08, "lambda_1":1e-08, "lambda_2":1.0, "fit_intercept":True},
    "KNN": {"n_neighbors":4, "p":1, "weights":"uniform"},
    "LASSO": {"alpha":0.001, "max_iter":10000, "random_state":33},
    "PLS": {"n_components":3, "scale":True},
    "SVR": {"kernel":"rbf", "C":0.3, "epsilon":0.0001, "gamma":0.0372759372031494},
    "RF": {"n_estimators":50, "max_depth":7, "max_features":0.3,
           "min_samples_leaf":2, "min_samples_split":4, "random_state":1, "n_jobs":-1},
    "XGB": {"objective":"reg:squarederror", "random_state":42, "n_jobs":-1,
            "n_estimators":400, "learning_rate":0.05, "max_depth":2,
            "min_child_weight":5, "subsample":0.7, "colsample_bytree":0.7, "verbosity":0},
    "LGBM": {"random_state":42, "verbose":-1, "n_estimators":500,
             "learning_rate":0.01, "max_depth":5, "min_child_samples":5,
             "num_leaves":15, "subsample":1.0, "n_jobs":-1},
    "Cat": {"random_seed":1, "verbose":0, "depth":6, "iterations":300,
            "l2_leaf_reg":1, "learning_rate":0.01}
}

def make_builders_for_BR(params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        core = Pipeline([("prep", pre), ("br", BayesianRidge(**params, tol=1e-6))])
        model = TransformedTargetRegressor(regressor=core, transformer=StandardScaler())
        model.fit(X_train, y_train)
        return model
    def create_unfitted():
        pre = create_preprocess()
        core = Pipeline([("prep", pre), ("br", BayesianRidge(**params, tol=1e-6))])
        return TransformedTargetRegressor(regressor=core, transformer=StandardScaler())
    return build_final, create_unfitted

def make_builders_for_pipeline_model(sklearn_model_cls, params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre), ("mdl", sklearn_model_cls(**params))])
        pipe.fit(X_train, y_train)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("mdl", sklearn_model_cls(**params))])
    return build_final, create_unfitted

def make_builders_for_pls(params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre), ("pls", PLSRegression(**params))])
        pipe.fit(X_train, y_train)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("pls", PLSRegression(**params))])
    return build_final, create_unfitted

def make_builders_for_lightgbm(params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        mdl = lgb.LGBMRegressor(**params)
        pipe = Pipeline([("prep", pre), ("lgbm", mdl)])
        pipe.fit(X_train, y_train)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("lgbm", lgb.LGBMRegressor(**params))])
    return build_final, create_unfitted

def make_builders_for_xgb(params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        mdl = XGBRegressor(**params)
        pipe = Pipeline([("prep", pre), ("xgb", mdl)])
        pipe.fit(X_train, y_train)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("xgb", XGBRegressor(**params))])
    return build_final, create_unfitted

def make_builders_for_cat(params):
    def build_final(X_train, y_train):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre),
                         ("cat", CatBoostRegressor(depth=params["depth"],
                                                   iterations=params["iterations"],
                                                   l2_leaf_reg=params["l2_leaf_reg"],
                                                   learning_rate=params["learning_rate"],
                                                   random_seed=params.get("random_seed",1),
                                                   verbose=0))])
        pipe.fit(X_train, y_train)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre),
                         ("cat", CatBoostRegressor(depth=params["depth"],
                                                   iterations=params["iterations"],
                                                   l2_leaf_reg=params["l2_leaf_reg"],
                                                   learning_rate=params["learning_rate"],
                                                   random_seed=params.get("random_seed",1),
                                                   verbose=0))])
    return build_final, create_unfitted

builders = {
    "BR": make_builders_for_BR(MODEL_PARAMS["BR"]),
    "KNN": make_builders_for_pipeline_model(KNeighborsRegressor, MODEL_PARAMS["KNN"]),
    "LASSO": make_builders_for_pipeline_model(Lasso, MODEL_PARAMS["LASSO"]),
    "PLS": make_builders_for_pls(MODEL_PARAMS["PLS"]),
    "SVR": make_builders_for_pipeline_model(SVR, MODEL_PARAMS["SVR"]),
    "RF": make_builders_for_pipeline_model(RandomForestRegressor, MODEL_PARAMS["RF"]),
    "XGB": make_builders_for_xgb(MODEL_PARAMS["XGB"]),
    "LGBM": make_builders_for_lightgbm(MODEL_PARAMS["LGBM"]),
    "Cat": make_builders_for_cat(MODEL_PARAMS["Cat"])
}

model_order = ["BR","KNN","LASSO","PLS","SVR","RF","XGB","LGBM","Cat"]

results = []
start_all = time.time()

for name in model_order:
    print("\n" + "="*60)
    print(f"Evaluating model: {name}")
    t0 = time.time()

    build_final, create_unfitted = builders[name]

    # --- 1) Fit final model ---
    final_model = build_final(X_train_raw, y_train)

    pred_train = final_model.predict(X_train_raw)
    pred_test  = final_model.predict(X_test_raw)

    # --- 2) LOOCV ---
    preds_loocv = np.zeros_like(y_train, dtype=float)
    for tr_idx, vl_idx in LOO.split(X_train_raw):
        mdl = create_unfitted()
        mdl.fit(X_train_raw.iloc[tr_idx], y_train[tr_idx])
        preds_loocv[vl_idx] = mdl.predict(X_train_raw.iloc[vl_idx])

    # --- 3) Point estimates ---
    pt_train = metrics_tuple(y_train, pred_train)
    pt_loocv  = metrics_tuple(y_train, preds_loocv)
    pt_test  = metrics_tuple(y_test, pred_test)

    print("\n--- Point estimates ---")
    pretty_point("Train", y_train, pred_train)
    pretty_point("LOOCV", y_train, preds_loocv)
    pretty_point("Test",  y_test,  pred_test)

    # --- 4) Bootstrap CI ---
    print(f"\nComputing bootstrap 95% CI (B={B})...")
    ci_train = paired_bootstrap_ci(y_train, pred_train, B=B, seed=SEED_BASE)
    ci_loocv = paired_bootstrap_ci(y_train, preds_loocv, B=B, seed=SEED_BASE+1)
    ci_test  = paired_bootstrap_ci(y_test, pred_test,  B=B, seed=SEED_BASE+2)

    print("\n--- Bootstrap 95% CI ---")
    pretty_with_ci("Train", pt_train, ci_train)
    pretty_with_ci("LOOCV", pt_loocv, ci_loocv)
    pretty_with_ci("Test", pt_test, ci_test)

    elapsed = time.time() - t0
    print(f"\nModel {name} done in {elapsed:.1f}s")

    results.append({
        "model": name,
        "pt_train": pt_train, "ci_train": ci_train,
        "pt_loocv": pt_loocv, "ci_loocv": ci_loocv,
        "pt_test": pt_test,  "ci_test": ci_test,
        "time_s": elapsed
    })

total_elapsed = time.time() - start_all
print("\n" + "="*60)
print(f"All models finished in {total_elapsed:.1f}s")

rows = []
for r in results:
    def ci_to_str(ci):
        return f"R2[{ci['r2_ci'][0]:.3f},{ci['r2_ci'][1]:.3f}] RMSE[{ci['rmse_ci'][0]:.3f},{ci['rmse_ci'][1]:.3f}] MAE[{ci['mae_ci'][0]:.3f},{ci['mae_ci'][1]:.3f}]"
    rows.append({
        "model": r["model"],
        "train_R2": r["pt_train"][0], "train_RMSE": r["pt_train"][1], "train_MAE": r["pt_train"][2],
        "loocv_R2": r["pt_loocv"][0], "loocv_RMSE": r["pt_loocv"][1], "loocv_MAE": r["pt_loocv"][2],
        "test_R2": r["pt_test"][0], "test_RMSE": r["pt_test"][1], "test_MAE": r["pt_test"][2],
        "CI_Train": ci_to_str(r["ci_train"]),
        "CI_LOOCV": ci_to_str(r["ci_loocv"]),
        "CI_Test":  ci_to_str(r["ci_test"]),
        "time_s": round(r["time_s"],2)
    })

summary_df = pd.DataFrame(rows).set_index("model")
pd.set_option("display.max_colwidth", 200)
print("\n=== Summary table ===")
display(summary_df)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
from copy import deepcopy
from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import TransformedTargetRegressor
from sklearn.linear_model import LinearRegression

# Models
from sklearn.linear_model import BayesianRidge, Lasso
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
import lightgbm as lgb
from catboost import CatBoostRegressor

TRAIN_CSV = "train.csv"
TEST_CSV  = "test.csv"

B = 1000               
SEED_BASE = 42       
LOO = LeaveOneOut()

def rmse(y_true, y_pred):
    return mean_squared_error(y_true, y_pred, squared=False)

def metrics_tuple(y_true, y_pred):
    return (r2_score(y_true, y_pred), rmse(y_true, y_pred), mean_absolute_error(y_true, y_pred))

def paired_bootstrap_ci(y_true, y_pred, B=B, seed=SEED_BASE):
    """Paired bootstrap (resample indices) returning dict of 95% CI arrays."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    r2s, rmses, maes = [], [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        yt = y_true[idx]
        yp = y_pred[idx]
        r2s.append(r2_score(yt, yp))
        rmses.append(mean_squared_error(yt, yp, squared=False))
        maes.append(mean_absolute_error(yt, yp))
    return {
        "r2_ci": np.percentile(r2s, [2.5, 97.5]),
        "rmse_ci": np.percentile(rmses, [2.5, 97.5]),
        "mae_ci": np.percentile(maes, [2.5, 97.5])
    }

def pretty_pt(tag, y_true, y_pred):
    r2, rrmse, rmae = metrics_tuple(y_true, y_pred)
    print(f"{tag:<6}| R²={r2:.3f}  RMSE={rrmse:.3f}  MAE={rmae:.3f}")
    return (r2, rrmse, rmae)

def pretty_ci(tag, pt, ci):
    r2, rrmse, rmae = pt
    r2_l, r2_h = ci["r2_ci"]
    rmse_l, rmse_h = ci["rmse_ci"]
    mae_l, mae_h = ci["mae_ci"]
    print(f"{tag:<6}| R²={r2:.3f} (95% CI {r2_l:.3f}–{r2_h:.3f}) | "
          f"RMSE={rrmse:.3f} (95% CI {rmse_l:.3f}–{rmse_h:.3f}) | "
          f"MAE={rmae:.3f} (95% CI {mae_l:.3f}–{mae_h:.3f})")

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)

y_train = train_df["Liver_regeneration_rate"].values
y_test  = test_df["Liver_regeneration_rate"].values

X_train_raw = train_df.drop(columns=["Liver_regeneration_rate"])
X_test_raw  = test_df.drop(columns=["Liver_regeneration_rate"])

# columns
cat_cols = ["Time_points"]
num_cols = [c for c in X_train_raw.columns if c not in cat_cols]

def create_preprocess():
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    num_pipe = Pipeline([
        ("log1p", FunctionTransformer(np.log1p, validate=False)),
        ("scale", StandardScaler())
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", ohe,      cat_cols)
    ])

MODEL_PARAMS = {
    "BR": {"alpha_1":1.0, "alpha_2":1e-08, "lambda_1":1e-08, "lambda_2":1.0, "fit_intercept":True},
    "KNN": {"n_neighbors":4, "p":1, "weights":"uniform"},
    "LASSO": {"alpha":0.001, "max_iter":10000, "random_state":33},
    "PLS": {"n_components":3, "scale":True},
    "SVR": {"kernel":"rbf", "C":0.3, "epsilon":0.0001, "gamma":0.0372759372031494},
    "RF": {"n_estimators":50, "max_depth":7, "max_features":0.3,
           "min_samples_leaf":2, "min_samples_split":4, "random_state":1, "n_jobs":-1},
    "XGB": {"objective":"reg:squarederror", "random_state":42, "n_jobs":-1,
            "n_estimators":400, "learning_rate":0.05, "max_depth":2,
            "min_child_weight":5, "subsample":0.7, "colsample_bytree":0.7, "verbosity":0},
    "LGBM": {"random_state":42, "verbose":-1, "n_estimators":500,
             "learning_rate":0.01, "max_depth":5, "min_child_samples":5,
             "num_leaves":15, "subsample":1.0, "n_jobs":-1},
    "Cat": {"random_seed":1, "verbose":0, "depth":6, "iterations":300,
            "l2_leaf_reg":1, "learning_rate":0.01}
}

def make_builders_for_BR(params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        core = Pipeline([("prep", pre), ("br", BayesianRidge(**params, tol=1e-6))])
        model = TransformedTargetRegressor(regressor=core, transformer=StandardScaler())
        model.fit(X_tr, y_tr)
        return model
    def create_unfitted():
        pre = create_preprocess()
        core = Pipeline([("prep", pre), ("br", BayesianRidge(**params, tol=1e-6))])
        return TransformedTargetRegressor(regressor=core, transformer=StandardScaler())
    return build_final, create_unfitted

def make_builders_for_pipeline_model(sklearn_cls, params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre), ("mdl", sklearn_cls(**params))])
        pipe.fit(X_tr, y_tr)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("mdl", sklearn_cls(**params))])
    return build_final, create_unfitted

def make_builders_for_pls(params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre), ("pls", PLSRegression(**params))])
        pipe.fit(X_tr, y_tr)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("pls", PLSRegression(**params))])
    return build_final, create_unfitted

def make_builders_for_lightgbm(params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        mdl = lgb.LGBMRegressor(**params)
        pipe = Pipeline([("prep", pre), ("lgbm", mdl)])
        pipe.fit(X_tr, y_tr)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("lgbm", lgb.LGBMRegressor(**params))])
    return build_final, create_unfitted

def make_builders_for_xgb(params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        mdl = XGBRegressor(**params)
        pipe = Pipeline([("prep", pre), ("xgb", mdl)])
        pipe.fit(X_tr, y_tr)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre), ("xgb", XGBRegressor(**params))])
    return build_final, create_unfitted

def make_builders_for_cat(params):
    def build_final(X_tr, y_tr):
        pre = create_preprocess()
        pipe = Pipeline([("prep", pre),
                         ("cat", CatBoostRegressor(depth=params["depth"],
                                                   iterations=params["iterations"],
                                                   l2_leaf_reg=params["l2_leaf_reg"],
                                                   learning_rate=params["learning_rate"],
                                                   random_seed=params.get("random_seed",1),
                                                   verbose=0))])
        pipe.fit(X_tr, y_tr)
        return pipe
    def create_unfitted():
        pre = create_preprocess()
        return Pipeline([("prep", pre),
                         ("cat", CatBoostRegressor(depth=params["depth"],
                                                   iterations=params["iterations"],
                                                   l2_leaf_reg=params["l2_leaf_reg"],
                                                   learning_rate=params["learning_rate"],
                                                   random_seed=params.get("random_seed",1),
                                                   verbose=0))])
    return build_final, create_unfitted

builders = {
    "BR": make_builders_for_BR(MODEL_PARAMS["BR"]),
    "KNN": make_builders_for_pipeline_model(KNeighborsRegressor, MODEL_PARAMS["KNN"]),
    "LASSO": make_builders_for_pipeline_model(Lasso, MODEL_PARAMS["LASSO"]),
    "PLS": make_builders_for_pls(MODEL_PARAMS["PLS"]),
    "SVR": make_builders_for_pipeline_model(SVR, MODEL_PARAMS["SVR"]),
    "RF": make_builders_for_pipeline_model(RandomForestRegressor, MODEL_PARAMS["RF"]),
    "XGB": make_builders_for_xgb(MODEL_PARAMS["XGB"]),
    "LGBM": make_builders_for_lightgbm(MODEL_PARAMS["LGBM"]),
    "Cat": make_builders_for_cat(MODEL_PARAMS["Cat"])
}

model_order = ["BR","KNN","LASSO","PLS","SVR","RF","XGB","LGBM","Cat"]

results = []
t_all0 = time.time()
for name in model_order:
    print("\n" + "="*60)
    print(f"Evaluating model: {name}")
    t0 = time.time()

    build_final, create_unfitted = builders[name]

    # 1) train final model on full training set
    final_model = build_final(X_train_raw, y_train)

    # predictions arrays (note: keep order consistent)
    pred_train = final_model.predict(X_train_raw)
    pred_test  = final_model.predict(X_test_raw)

    # 2) LOOCV manual (create_unfitted returns a fresh model pipeline / transformed-target for BR)
    preds_loocv = np.zeros_like(y_train, dtype=float)
    for tr_idx, vl_idx in LOO.split(X_train_raw):
        mdl = create_unfitted()
        mdl.fit(X_train_raw.iloc[tr_idx], y_train[tr_idx])
        preds_loocv[vl_idx] = mdl.predict(X_train_raw.iloc[vl_idx])

    # 3) point estimates
    pt_train = metrics_tuple(y_train, pred_train)
    pt_loocv  = metrics_tuple(y_train, preds_loocv)
    pt_test  = metrics_tuple(y_test, pred_test)

    print("\n--- Point estimates ---")
    pretty_pt("Train", y_train, pred_train)
    pretty_pt("LOOCV", y_train, preds_loocv)
    pretty_pt("Test",  y_test,  pred_test)

    # 4) bootstrap CI (paired)
    print(f"\nComputing bootstrap 95% CI (B={B}) ... (this may take a while)")
    ci_train = paired_bootstrap_ci(y_train, pred_train, B=B, seed=SEED_BASE)
    ci_loocv = paired_bootstrap_ci(y_train, preds_loocv, B=B, seed=SEED_BASE+1)
    ci_test  = paired_bootstrap_ci(y_test, pred_test,  B=B, seed=SEED_BASE+2)

    print("\n--- Bootstrap 95% CI ---")
    pretty_ci("Train", pt_train, ci_train)
    pretty_ci("LOOCV", pt_loocv, ci_loocv)
    pretty_ci("Test", pt_test, ci_test)

    elapsed = time.time() - t0
    print(f"\nModel {name} done in {elapsed:.1f}s")

    results.append({
        "model": name,
        "pred_train": pred_train,
        "pred_loocv": preds_loocv,
        "pred_test": pred_test,
        "pt_train": pt_train, "ci_train": ci_train,
        "pt_loocv": pt_loocv, "ci_loocv": ci_loocv,
        "pt_test": pt_test,  "ci_test": ci_test,
        "time_s": elapsed
    })

print("\n" + "="*60)
print(f"All base models evaluated in {time.time()-t_all0:.1f}s")

rows = []
for r in results:
    m = r["model"]
    tr2, trrmse, trmae = r["pt_train"]
    lo2, lormse, lomae = r["pt_loocv"]
    te2, termse, temae = r["pt_test"]
    rows.append({
        "model": m,
        "train_R2": tr2, "train_RMSE": trrmse, "train_MAE": trmae,
        "loocv_R2": lo2, "loocv_RMSE": lormse, "loocv_MAE": lomae,
        "test_R2": te2, "test_RMSE": termse, "test_MAE": temae
    })
summary_df = pd.DataFrame(rows).set_index("model")
print("\n=== Using Test set metrics for multi-model comparison ===")
display(summary_df[["test_R2","test_RMSE","test_MAE"]])

val_array = summary_df[["test_R2","test_RMSE","test_MAE"]].values
converted = np.column_stack([-val_array[:,0], val_array[:,1], val_array[:,2]])

def is_dominated(point, others):
    for o in others:
        if np.all(o <= point) and np.any(o < point):
            return True
    return False

pareto_mask = []
for i, p in enumerate(converted):
    others = np.delete(converted, i, axis=0)
    pareto_mask.append(not is_dominated(p, others))

pareto_models = summary_df.index[np.array(pareto_mask)]
print("\n=== Pareto model ===")
print(list(pareto_models))
display(summary_df.loc[pareto_models, ["test_R2","test_RMSE","test_MAE"]])

def topsis(df_metrics, weights=None, maximize=[True, False, False]):
    """
    df_metrics: DataFrame, rows=models, cols=metrics in a fixed order (e.g. [R2,RMSE,MAE])
    maximize: boolean list same length: True if metric is benefit (larger better)
    weights: list or None (if None equal weights)
    Returns DataFrame with closeness values and rankings (higher closeness better)
    """
    X = df_metrics.values.astype(float)
    n_obj = X.shape[1]
    if weights is None:
        weights = np.ones(n_obj) / n_obj
    else:
        weights = np.array(weights) / np.sum(weights)

    sign = np.array([1 if m else -1 for m in maximize])
    X_signed = X * sign  # now larger is better for all

    norm = np.sqrt((X_signed**2).sum(axis=0))
    norm[norm == 0] = 1.0
    X_norm = X_signed / norm

    V = X_norm * weights

    ideal_best = V.max(axis=0)
    ideal_worst = V.min(axis=0)

    dist_pos = np.sqrt(((V - ideal_best)**2).sum(axis=1))
    dist_neg = np.sqrt(((V - ideal_worst)**2).sum(axis=1))

    closeness = dist_neg / (dist_pos + dist_neg + 1e-12)  
    out = pd.DataFrame({
        "closeness": closeness
    }, index=df_metrics.index)
    out = out.sort_values("closeness", ascending=False)
    return out

pareto_df = summary_df.loc[pareto_models, ["test_R2","test_RMSE","test_MAE"]]
topsis_pareto = topsis(pareto_df, weights=None, maximize=[True, False, False])
print("\n=== TOPSIS on Pareto set (equal weights) ===")
display(topsis_pareto)

all_df = summary_df[["test_R2","test_RMSE","test_MAE"]]
topsis_all = topsis(all_df, weights=None, maximize=[True, False, False])
print("\n=== TOPSIS on ALL models (equal weights) ===")
display(topsis_all)

order = list(topsis_all.index)
print("\n（TOPSIS, high -> low）:", order)

meta_train_cols = []
meta_test_cols = []
for r in results:
    meta_train_cols.append(r["pred_loocv"])
    meta_test_cols.append(r["pred_test"])

name_to_preds = {r["model"]: {"loocv": r["pred_loocv"], "test": r["pred_test"]} for r in results}

from sklearn.linear_model import LinearRegression

stacking_records = []
print("\n=== stacking（rank on TOPSIS） ===\n")
for k in range(1, len(order)+1):
    sel_names = order[:k]
    X_meta_train = np.column_stack([name_to_preds[n]["loocv"] for n in sel_names])
    X_meta_test  = np.column_stack([name_to_preds[n]["test"]  for n in sel_names])

    meta = LinearRegression()
    meta.fit(X_meta_train, y_train)

    pred_meta_train = meta.predict(X_meta_train)
    pred_meta_test  = meta.predict(X_meta_test)

    tr_r2, tr_rmse, tr_mae = metrics_tuple(y_train, pred_meta_train)
    te_r2, te_rmse, te_mae = metrics_tuple(y_test,  pred_meta_test)

    print(f">>> 组合 {k}: {sel_names}")
    print(f"Train R²={tr_r2:.3f}  RMSE={tr_rmse:.3f} MAE={tr_mae:.3f}")
    print(f"Test  R²={te_r2:.3f}  RMSE={te_rmse:.3f} MAE={te_mae:.3f}\n")

    stacking_records.append({
        "members": "+".join(sel_names),
        "train_R2": tr_r2, "train_RMSE": tr_rmse, "train_MAE": tr_mae,
        "test_R2": te_r2,  "test_RMSE": te_rmse,  "test_MAE": te_mae
    })

stacking_df = pd.DataFrame(stacking_records)
print("\n=== Incremental stacking summary (sorted by TOPSIS and models added step by step) ===")
display(stacking_df)

best_idx = stacking_df["test_R2"].idxmax()
print("\nBest stacking combination by Test R²:")
display(stacking_df.loc[[best_idx]])

print("\nScript finished.")


In [ ]:
print("\n\n" + "="*80)
print("=== Base + Stacking all models TOPSIS rank（Equal Weights） ===")
all_models_records = []

for name in summary_df.index:
    all_models_records.append({
        "model": name,
        "R2": summary_df.loc[name, "test_R2"],
        "RMSE": summary_df.loc[name, "test_RMSE"],
        "MAE": summary_df.loc[name, "test_MAE"]
    })

for i, row in stacking_df.iterrows():
    all_models_records.append({
        "model": f"STACK_{row['members']}",
        "R2": row["test_R2"],
        "RMSE": row["test_RMSE"],
        "MAE": row["test_MAE"]
    })

all_models_df = pd.DataFrame(all_models_records).set_index("model")

topsis_full = topsis(
    df_metrics=all_models_df[["R2","RMSE","MAE"]],
    weights=None,
    maximize=[True, False, False]
)

print("\n=== TOPSIS：Single model + all stacking combinations ===")
display(topsis_full)
print("\n=== TOPSIS Total top 10 rankings ===")
display(topsis_full.head(10))
best_model = topsis_full.index[0]
print(f"\n>>> TOPSIS optimal model among all models： {best_model}")
print("\n追加模块执行完毕。")


In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

selected_learners = ["PLS", "BR", "LASSO", "KNN"]
top_n = 5

save_dir = Path("./shap_pdf")
save_dir.mkdir(exist_ok=True)

bg = X_train_raw.sample(n=min(80, len(X_train_raw)), random_state=1)

for name in selected_learners:
    print(f"\n--- Processing model：{name} ---")

    build_final, _ = builders[name]
    model = build_final(X_train_raw, y_train)

    if name == "BR":
        preproc = model.regressor_.named_steps["prep"]
        inner = model.regressor_.named_steps["br"]
    else:
        preproc = model.named_steps["prep"]
        inner = model.named_steps[list(model.named_steps.keys())[1]]

    X_bg_full = preproc.transform(bg)
    X_full = preproc.transform(X_train_raw)
    explainer = shap.KernelExplainer(inner.predict, X_bg_full)
    idx = np.random.choice(X_full.shape[0], size=min(120, X_full.shape[0]), replace=False)
    shap_vals_full = explainer.shap_values(X_full[idx])
    metab_shap = shap_vals_full[:, :len(num_cols)]
    metab_data = X_full[idx][:, :len(num_cols)]
    mean_abs = np.abs(metab_shap).mean(axis=0)
    top_idx = np.argsort(mean_abs)[-top_n:][::-1]

    top_features = np.array(num_cols)[top_idx]
    shap_top = metab_shap[:, top_idx]
    X_top = metab_data[:, top_idx]
    plt.figure(figsize=(6, 4))
    shap.summary_plot(shap_top, X_top, feature_names=top_features,
                      plot_type="bar", show=False)

    save_path = save_dir / f"SHAP_{name}_top5.pdf"
    plt.savefig(save_path, format="pdf", bbox_inches="tight")
    plt.close()

    print(f"PDF saved：{save_path}")
    plt.figure(figsize=(6, 4))
    shap.summary_plot(shap_top, X_top, feature_names=top_features,
                      plot_type="bar", show=True)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi

data = {
    "Model": ["KNN", "LASSO", "BR", "PLS", "Stacking"],
    "R2":    [0.859, 0.863, 0.867, 0.899, 0.922],
    "RMSE":  [0.255, 0.252, 0.248, 0.217, 0.190],
    "MAE":   [0.207, 0.204, 0.205, 0.155, 0.157]
}

df = pd.DataFrame(data)

model_colors = {
    "KNN": "#EFCD74",      
    "LASSO": "#F89F65",    
    "BR": "#D55D1E",       
    "PLS": "#889851",      
    "Stacking": "#37604E"  
}

plt.figure(figsize=(8,5))

df_melt = df.melt(id_vars="Model", var_name="Metric", value_name="Value")

palette = [model_colors[m] for m in df["Model"]]

sns.barplot(
    data=df_melt,
    x="Metric",
    y="Value",
    hue="Model",
    palette=palette
)

plt.title("Performance of 5 Models on Test Set")
plt.ylabel("Value")
plt.xlabel("Metric")
plt.legend(title="Model", fontsize=10)

plt.tight_layout()
plt.savefig("barplot_models_customcolor.pdf", dpi=600)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Polygon, Circle
from matplotlib import rcParams
import matplotlib.patheffects as path_effects

rcParams["font.family"] = "Times New Roman"
rcParams["font.size"] = 14
rcParams["font.weight"] = "bold" 
models = ["KNN", "LASSO", "BR", "PLS", "Stacking"]
metrics = ["R2", "RMSE", "MAE"]

data = {
    "PLS": [0.899, 0.217, 0.155],
    "BR": [0.867, 0.248, 0.205],
    "LASSO": [0.863, 0.252, 0.204],
    "KNN": [0.859, 0.255, 0.207],
    "Stacking": [0.922, 0.190, 0.157]
}

model_colors = {
    "KNN": "#EFCD74",
    "LASSO": "#F89F65",
    "BR": "#D55D1E",
    "PLS": "#889851",
    "Stacking": "#37604E"
}

fill_alpha = 0.20
line_width = 2.2
point_size = 60
grid_color = "#AAAAAA"
r2_min, r2_max = 0.85, 0.95
rmse_vals = [v[1] for v in data.values()]
mae_vals  = [v[2] for v in data.values()]
rmse_min, rmse_max = min(rmse_vals), max(rmse_vals)
mae_min, mae_max   = min(mae_vals), max(mae_vals)

norm_data = {}
for m, (r2, rmse, mae) in data.items():
    r2_n = (r2 - r2_min) / (r2_max - r2_min)
    rmse_n = 1 - (rmse - rmse_min) / (rmse_max - rmse_min)
    mae_n = 1 - (mae - mae_min) / (mae_max - mae_min)
    norm_data[m] = [r2_n, rmse_n, mae_n]
N = 3
angles = np.linspace(0, 2*np.pi, N, endpoint=False)
angles_closed = np.append(angles, angles[0])

outer_x = np.sin(angles_closed)
outer_y = np.cos(angles_closed)
fig, ax = plt.subplots(figsize=(8,8), facecolor="#f7f7f2")
ax.set_aspect('equal')
ax.axis('off')
inner_circle = Circle((0,0), 1.02, facecolor="#ffffff", edgecolor="none", zorder=0)
ax.add_patch(inner_circle)
outer_circle = Circle(
    (0,0), 1.02,
    edgecolor=grid_color,
    facecolor="none",
    linewidth=1.0,
    alpha=0.9,
    zorder=2
)
ax.add_patch(outer_circle)
grid_levels = np.linspace(0.2, 1.0, 5)
for r in grid_levels:
    gx = r * outer_x
    gy = r * outer_y
    ax.plot(gx, gy, color=grid_color, linewidth=0.9, alpha=0.6, zorder=3)

for ang in angles:
    ax.plot([0, np.sin(ang)], [0, np.cos(ang)],
            color=grid_color, linewidth=1.0, alpha=0.6, zorder=3)

label_r = 1.13
for lab, ang in zip(metrics, angles):
    ax.text(label_r*np.sin(ang), label_r*np.cos(ang),
            lab, ha="center", va="center", fontsize=15, zorder=4)

shadow = [
    path_effects.SimpleLineShadow(offset=(1, -1), alpha=0.25),
    path_effects.Normal()
]

for model in models:
    vals_norm = norm_data[model]
    vals_close = np.append(vals_norm, vals_norm[0])
    xs = vals_close * np.sin(angles_closed)
    ys = vals_close * np.cos(angles_closed)

    ax.add_patch(Polygon(
        np.column_stack([xs, ys]),
        closed=True,
        facecolor=model_colors[model],
        alpha=fill_alpha,
        edgecolor=None,
        zorder=5
    ))

    ax.plot(xs, ys, color=model_colors[model],
            linewidth=line_width, zorder=6,
            path_effects=shadow)

    ax.scatter(xs[:-1], ys[:-1], s=point_size,
               color=model_colors[model], zorder=7)

    if model in ["Stacking", "PLS", "BR"]:
        offset = {"Stacking": 0.04, "PLS": 0.03, "BR": 0.045}[model]

        for ang, rn, raw in zip(angles, vals_norm, data[model]):
            ax.annotate(
                f"{raw:.3f}",
                xy=(rn*np.sin(ang), rn*np.cos(ang)),
                xytext=(rn*np.sin(ang), rn*np.cos(ang) + offset),
                fontsize=11, fontweight="bold",
                color=model_colors[model],
                ha="center",
                arrowprops=dict(
                    arrowstyle="-", color=model_colors[model],
                    alpha=0.5, lw=1
                ),
                zorder=10
            )

handles = [
    plt.Line2D([0],[0], color=model_colors[m], linewidth=2.5,
               marker="o", markersize=6, label=m)
    for m in models
]
ax.legend(handles=handles, loc="upper right", fontsize=12, frameon=False)

plt.savefig("radar_final.pdf", dpi=600, bbox_inches="tight", facecolor="white")
plt.savefig("radar_final.png", dpi=600, bbox_inches="tight", facecolor="white")

plt.show()
